In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

from principled_pclm.sim import SimPoisson1D, make_aggregation_matrix
from principled_pclm.models._inference import run_inference_mcmc

from principled_pclm.models import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_composition_matrix,
)

# For visualization
import altair as alt

## Utility functions

In [ ]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(
    df_obs: pd.DataFrame,
    df_rates: pd.DataFrame,
    show_empirical: bool = True,
    show_legend: bool = True,
    color_scale: alt.Scale | None = None,
) -> alt.LayerChart:
    """
    Plot the true rates, the average rates over the observed intervals, and optionally the empirical rates from the observed data.

    Parameters
    ----------
    df_obs : pd.DataFrame
        DataFrame containing the observed data with columns 'obs_interval', 'x', and 'y'.
    df_rates : pd.DataFrame
        DataFrame containing the true rates with columns 'x' and 'lambda'.
    show_empirical : bool, optional
        If True, include the empirical rates in the plot. Default is True.
    show_legend : bool, optional
        If True, show a legend labeling each series. Default is True.
    color_scale : alt.Scale | None, optional
        Color scale to use for the "series" field, keyed by "True rate",
        "Interval average", and (if `show_empirical`) "Empirical rate". If
        None (default), a scale covering just those series is built here.
        Pass an explicit scale when layering this chart's output with other
        charts that share the "series" color field -- e.g. `plot_posterior`
        -- since Vega-Lite only keeps the *first* layer's `range` when
        merging per-layer color scales that declare different `range`
        arrays, silently reusing colors for domain values added by later
        layers otherwise.

    Returns
    -------
    alt.LayerChart
                An Altair chart object containing the plot.

    See Also
    --------
    PoissonEnveloped1D
    """

    df_segment = expand_interval_dataframe(
        df_obs, interval_col="obs_interval", x_col="x"
    )
    source_segment = summarize_interval_segments(df_segment)

    if color_scale is None:
        legend_domain = ["True rate", "Interval average"]
        legend_range = ["#de425b", "#594e90"]
        if show_empirical:
            legend_domain.append("Empirical rate")
            legend_range.append("#ffa600")
        color_scale = alt.Scale(domain=legend_domain, range=legend_range)
    color_legend = alt.Legend(title=None) if show_legend else None

    chart_true = (
        alt.Chart(df_rates.assign(series="True rate"))
        .mark_line()
        .encode(
            x=alt.X("x:Q", title="x"),
            y=alt.Y("lambda:Q", title="λ"),
            color=alt.Color(
                "series:N", scale=color_scale, title=None, legend=color_legend
            ),
        )
    )

    chart_segment = (
        alt.Chart(source_segment.assign(series="Interval average"))
        .mark_rule(strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q", title="x"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="λ"),
            color=alt.Color(
                "series:N", scale=color_scale, title=None, legend=color_legend
            ),
        )
    )

    if show_empirical:
        df_emp = df_obs.groupby("obs_interval")["y"].agg(["sum", "count"]).reset_index()
        df_emp["lambda"] = df_emp["sum"] / df_emp["count"]
        df_emp = expand_interval_dataframe(
            df_emp, interval_col="obs_interval", x_col="x"
        )
        source_emp = summarize_interval_segments(df_emp)

        chart_emp = (
            alt.Chart(source_emp.assign(series="Empirical rate"))
            .mark_rule(strokeWidth=2.5, opacity=0.85)
            .encode(
                x=alt.X("x_min:Q", title="x"),
                x2="x_max:Q",
                y=alt.Y("lambda_obs:Q", title="λ"),
                color=alt.Color(
                    "series:N", scale=color_scale, title=None, legend=color_legend
                ),
            )
        )

        return chart_true + chart_segment + chart_emp
    else:
        return chart_true + chart_segment

## Setup Block

In [10]:
# Input space
x = np.linspace(0, 100, 100)

# Intervals
# Size 5 intervals
left_limits = list(range(0, 100, 5))
right_limits = [i + 4 for i in left_limits]

# Size 10 intervals
# left_limits = list(range(0, 100, 10))
# right_limits = [i + 9 for i in left_limits]

# Size 20 intervals
# left_limits = list(range(0, 100, 20))
# right_limits = [i + 19 for i in left_limits]

### Generate Data

In [13]:
intervals = [f"[{left},{right}]" for left, right in zip(left_limits, right_limits)]
print("Intervals:", intervals)

df_obs, df_rates = SimPoisson1D(
    obs_intervals=intervals,
    n_observations=10,
    lenscale=10,
    marg_var=1.0,
    beta0=3.0,
    seed=1,
).generate(return_true_rates=True)

Intervals: ['[0,4]', '[5,9]', '[10,14]', '[15,19]', '[20,24]', '[25,29]', '[30,34]', '[35,39]', '[40,44]', '[45,49]', '[50,54]', '[55,59]', '[60,64]', '[65,69]', '[70,74]', '[75,79]', '[80,84]', '[85,89]', '[90,94]', '[95,99]']


In [14]:
plot_data(df_obs, df_rates, show_empirical=True)

alt.LayerChart(...)

### Generate Key Objects

In [ ]:
C = make_aggregation_matrix(left_limits, right_limits)

## Result Plotting Helpers

Shared helpers for the 4 downstream experiments' result plots, so each experiment's own plot and the final side-by-side comparison are built from the same code.

In [62]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior(
    source: pd.DataFrame,
    df_obs: pd.DataFrame,
    df_rates: pd.DataFrame,
    title: str | None = None,
) -> alt.LayerChart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    color_scale = alt.Scale(
        domain=["True rate", "Interval average", "Empirical rate", "Posterior mean"],
        range=["#de425b", "#594e90", "#ffa600", "purple"],
    )
    chart = (
        alt.Chart(source.assign(series="Posterior mean"))
        .mark_line()
        .encode(
            x="x",
            y="lam",
            color=alt.Color("series:N", scale=color_scale, title=None),
        )
    )
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_obs, df_rates, color_scale=color_scale) + chart + band
    return result.properties(title=title) if title else result

In [63]:
def summarize_posterior_lam_betaIm_median(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
) -> pd.DataFrame:
    """
    Posterior median of `lam` computed from `betaIm` only (the identified
    subspace), with the uncertainty band taken from the full posterior `lam`
    -- which comes from jointly fitting `betaIm` and `betaKer` via MCMC, so
    the null-space contribution to the band is data-informed rather than a
    raw prior draw. (A prior-only draw of `betaKer` produced implausibly
    wide bands, suggesting the null space isn't actually as uninformed as
    the Fisher-information analysis assumed -- jointly fitting it lets the
    likelihood constrain it instead.)
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]

    lam_Im = jnp.exp(beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T)).T)
    central = np.median(lam_Im, axis=0)

    lam_full = mcmc_samples["lam"]
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

In [64]:
def summarize_posterior_lam_nullspace_prior_uncertainty(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
    rng_key: ArrayLike,
    scale: float = 1.0,
) -> pd.DataFrame:
    """
    Posterior median of `lam` from betaIm only (`mcmc_samples["lam"]`, from a
    model_reduced_bspline run with no betaKer), with an uncertainty band
    built by drawing one fresh betaKer ~ Normal(0, I) per posterior betaIm
    sample and pushing it through Phi_eff -- i.e. null-space uncertainty
    taken from its prior, not fit to data.
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]
    S = beta0.shape[0]
    MKer = PrKer.shape[1]

    betaKer_prior = jax.random.normal(rng_key, shape=(S, MKer)) * scale
    lam_full = jnp.exp(
        beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T + PrKer @ betaKer_prior.T)).T
    )

    central = np.median(mcmc_samples["lam"], axis=0)
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

### Experiment 1: Naive B-Spline model

In [65]:
def model_bspline(
    C: jax.Array,
    Phi: jax.Array,
    N: jax.Array,
    id: jax.Array,
    K: jax.Array | None = None,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    C : jax.Array
        Composition matrix mapping fine-scale rates to observed intervals.
    Phi : jax.Array
        B-spline basis matrix.
    N : jax.Array
        Number of observations in each interval.
    id : jax.Array
        Interval indices for each observation.
    K : jax.Array | None, optional
        Cholesky factor of the coefficient covariance prior (Sigma = K @ K.T).
    y : jax.Array, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = Phi.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    beta = numpyro.sample("w", dist.Normal(0, 1).expand([M]).to_event(1))
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ beta))
    lam_interval = (
        jnp.matmul(C, lam) / C.sum(axis=1) * N[id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[id]), obs=y)

In [66]:
# Prepare the data for the model
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

# Convert to JAX arrays
model_kwargs = {
    "C": jnp.array(C),
    "Phi": jnp.array(Phi),
    "N": jnp.array(df_model["N"].values),
    "id": jnp.array(df_model["obs_interval_id"].values),
    "K": jnp.array(K),
    "y": jnp.array(df_model["y"].values),
}

In [67]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(rng_key_mcmc, model_bspline, **model_kwargs)
mcmc_samples = mcmc.get_samples()

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:04<00:00, 243.31it/s, 1023 steps of size 8.27e-04. acc. prob=0.94]

Number of divergences: 0


In [68]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior(source_exp1, df_obs, df_rates)

alt.LayerChart(...)